# Easy GRPO Kaggle Run

This notebook runs the shaped/easy GRPO curriculum for OnCallEnv Red Shift. It is separate from the hard Qwen2.5 3B GRPO notebook so the high-score easy result stays clearly labeled.

## 1. GPU Check

Expected on Kaggle: Tesla T4 GPUs. If this does not show GPUs, the notebook is not attached to the Kaggle GPU kernel.

In [2]:
!nvidia-smi

Sat Apr 25 12:05:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Bootstrap Repo

The Kaggle kernel cannot see the local laptop path. This clones or updates the pushed `round2-redshift` branch into `/kaggle/working`.

In [3]:
import os
from pathlib import Path

Path('/kaggle/working').mkdir(parents=True, exist_ok=True)
os.chdir('/kaggle/working')
print('cwd:', os.getcwd())


cwd: /kaggle/working


In [4]:
import os, shutil, subprocess, time
from pathlib import Path

REPO_URL = 'https://github.com/srimanreddy4/MetaHackathon-R2'
BRANCH = 'round2-redshift'
WORKDIR = Path('/kaggle/working/MetaHackathon-R2')

os.chdir('/kaggle/working')
if (WORKDIR / '.git').exists():
    os.chdir(WORKDIR)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    if WORKDIR.exists():
        backup = WORKDIR.with_name(f'{WORKDIR.name}.bak.{int(time.time())}')
        shutil.move(str(WORKDIR), str(backup))
        print('Moved non-git existing directory to', backup)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(WORKDIR)], check=True)
    os.chdir(WORKDIR)

print('cwd:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-5'], check=True)


Cloning into '/kaggle/working/MetaHackathon-R2'...


cwd: /kaggle/working/MetaHackathon-R2
82ae6b3 notebooks: add easy grpo kaggle run
3c5dbd6 training: add easy grpo curriculum mode
5de5a68 docs: summarize implementation and results
e3d8070 training: support interrupted grpo summaries
dcff16c updated notebook4


CompletedProcess(args=['git', 'log', '--oneline', '-5'], returncode=0)

In [6]:
%cd /kaggle/working/MetaHackathon-R2
%env PYTHONPATH=src:scripts
!python - <<'PY'
import os
from pathlib import Path
print('cwd=', Path.cwd())
print('PYTHONPATH=', os.environ.get('PYTHONPATH'))
print('repo exists=', Path('src/oncallenv').exists())



/kaggle/working
env: PYTHONPATH=src:scripts
/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
cwd= /kaggle/working/MetaHackathon-R2
PYTHONPATH= src:scripts
repo exists= True


## 3. Install Dependencies

Run once per Kaggle session.

In [7]:
!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements.txt
!python -m pip install -r requirements-llm.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.0 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.46.3
    Uninstalling wheel-0.46.3:
      Successfully uninstalled wheel-0.46.3
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 17.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [openenv-core] [openenv-core]tic]
INFO: pip is looking at multiple versions of unsloth to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of unsloth to dete

In [ ]:
# Fallback if dependency resolution fails:
# !python -m pip install -U transformers datasets accelerate trl peft bitsandbytes unsloth


## 4. Verify

Expected: `21 passed` and OpenEnv validation OK.

In [ ]:
!bash scripts/run_kaggle_qwen3b_grpo.sh verify

## 5. Easy Smoke Run

This validates easy prompts and shaped reward before spending more time. Expected score should be much higher than hard mode.

In [ ]:
!bash scripts/run_kaggle_qwen3b_grpo.sh easy-smoke

In [ ]:
!cat training_results/unsloth_grpo_qwen3b_easy_smoke/summary.json

## 6. Easy Main Run

This is the high-score shaped-curriculum run: easy prompts, dense partial-credit reward, 300 steps.

In [ ]:
!git pull
!bash scripts/run_kaggle_qwen3b_grpo.sh easy-main

remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 12 (delta 10), reused 12 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 2.93 KiB | 428.00 KiB/s, done.
From https://github.com/srimanreddy4/MetaHackathon-R2
   82ae6b3..5affb0b  round2-redshift -> origin/round2-redshift
   4f7b330..2adf149  spicy-attacker  -> origin/spicy-attacker
Updating 82ae6b3..5affb0b
Fast-forward
 scripts/run_kaggle_qwen3b_grpo.sh |  4 ++--
 scripts/train_unsloth_grpo.py     | 14 +++++++++-----
 2 files changed, 11 insertions(+), 7 deletions(-)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-04-25 12:20:38.970520: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777119638.993934     869 cuda_dnn.cc:8579] Unable to register cuD

In [ ]:
!cat training_results/unsloth_grpo_qwen3b_easy/summary.json
!bash scripts/run_kaggle_qwen3b_grpo.sh easy-summary

## 7. Optional Fallback

Use this only if the 3B easy run OOMs or behaves badly.

In [ ]:
# MODEL_NAME=unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit bash scripts/run_kaggle_qwen3b_grpo.sh easy-main

## 8. Archive Outputs

Download `/kaggle/working/qwen3b_grpo_results.tar.gz` from Kaggle outputs.

In [ ]:
!bash scripts/run_kaggle_qwen3b_grpo.sh archive